<a href="https://colab.research.google.com/github/YASHYOGESHAHIRE/03-dividend-based-investment-finance/blob/main/dividendbasedinvesting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#We are using nifty fifty, but nifty fifty dont give that much divident, so atleast we should give 500 to 1000 other stocks


In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
import math
from scipy import stats


In [3]:
tickers = pd.read_csv('/content/top_50_indian_stocks.csv')
tickers.head()

,Ticker,Company Name
0,RELIANCE.NS,Reliance Industries
1,TCS.NS,Tata Consultancy Services
2,HDFCBANK.NS,HDFC Bank
3,INFY.NS,Infosys
4,ICICIBANK.NS,ICICI Bank


In [4]:
## Weighted Scoring Model for finding the final Score


In [8]:
def create_dividend_df(tickers):
  columns=[
      "Ticker",
      "Dividend Yield(%)",
      "Dividend Rate",
      "Payout Ratio(%)",
      "5 Year Avg Dividend Yield(%)",
      "Earning Growth(%)"
  ]
  dividend_df = pd.DataFrame(columns=columns)
  for stock in tickers:
    ticker = yf.Ticker(stock)
    info = ticker.info

    # Corrected dividend_yield logic to handle 0.0 values correctly
    dividend_yield_raw = info.get("dividendYield")
    dividend_yield = dividend_yield_raw * 100 if dividend_yield_raw is not None else np.nan

    # These calculations already handle np.nan correctly due to default value in .get()
    dividend_rate = info.get("dividendRate", np.nan)
    payout_ratio = info.get("payoutRatio", np.nan) * 100
    five_year_avg_dividend_yield = info.get("fiveYearAvgDividendYield", np.nan) * 100
    earning_growth = info.get("earningsGrowth", np.nan)

    dividend_df.loc[len(dividend_df)] = [stock,dividend_yield,dividend_rate,payout_ratio,five_year_avg_dividend_yield,earning_growth]

  # Normalization should be done AFTER the dataframe is fully populated, so moved outside the loop
  numeric_cols=[
      "Dividend Yield(%)",
      "Dividend Rate",
      "Payout Ratio(%)",
      "5 Year Avg Dividend Yield(%)",
      "Earning Growth(%)"
  ]
  weights = {
      "Dividend Yield(%) Normalised":0.2,
      "Dividend Rate Normalised":0.2,
      "Payout Ratio(%) Normalised":0.2,
      "5 Year Avg Dividend Yield(%) Normalised":0.2,
      "Earning Growth(%) Normalised":0.2
  }

  # Apply normalization after collecting all data
  for col in numeric_cols:
    # Check if there are non-NaN values to avoid division by zero if all values are NaN
    if not dividend_df[col].isnull().all():
        col_min = dividend_df[col].min()
        col_max = dividend_df[col].max()

        if col_max - col_min == 0: # Handle cases where all non-NaN values are the same
            dividend_df[col+" Normalised"] = 0.5 # Assign a neutral value like 0.5
        # Corrected column name from "Payout Ratio" to "Payout Ratio(%)"
        elif col == "Payout Ratio(%)": # Payout ratio might be inverse
            dividend_df[col+" Normalised"] = 1 - (dividend_df[col]-col_min)/(col_max-col_min)
        else:
            dividend_df[col+" Normalised"] = (dividend_df[col]-col_min)/(col_max-col_min)
    else:
        dividend_df[col+" Normalised"] = np.nan # If all values are NaN, normalized is also NaN

  return dividend_df

In [9]:
tickers_list = tickers["Ticker"].values.tolist()
dividend_df = create_dividend_df(tickers_list)
dividend_df

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}


,Ticker,Dividend Yield(%),Dividend Rate,Payout Ratio(%),5 Year Avg Dividend Yield(%),Earning Growth(%),Dividend Yield(%) Normalised,Dividend Rate Normalised,Payout Ratio(%) Normalised,5 Year Avg Dividend Yield(%) Normalised,Earning Growth(%) Normalised
0,RELIANCE.NS,46.0,6.00,9.210000,44.0,-0.126,0.049238,0.034483,0.903083,0.047146,0.068275
1,TCS.NS,564.0,124.00,46.320000,149.0,0.122,0.656506,0.774295,0.512575,0.177419,0.091280
2,HDFCBANK.NS,174.0,13.00,21.760000,102.0,0.075,0.199297,0.078370,0.771020,0.119107,0.086920
3,INFY.NS,418.0,50.00,64.650000,240.0,0.118,0.485346,0.310345,0.319689,0.290323,0.090909
4,ICICIBANK.NS,87.0,11.00,14.710000,99.0,0.084,0.097304,0.065831,0.845207,0.115385,0.087755
5,HINDUNILVR.NS,207.0,44.00,95.030000,158.0,0.214,0.237984,0.272727,0.000000,0.188586,0.099814
6,SBIN.NS,177.0,17.35,17.440000,150.0,-0.031,0.202814,0.105643,0.816479,0.178660,0.077087
7,BAJFINANCE.NS,62.0,5.40,14.420000,24.0,0.214,0.067995,0.030721,0.848258,0.022333,0.099814
8,BHARTIARTL.NS,89.0,16.00,36.060002,80.0,-0.340,0.099648,0.097179,0.620541,0.091811,0.048423
9,ITC.NS,570.0,16.00,86.920000,372.0,-0.727,0.663540,0.097179,0.085341,0.454094,0.012523


In [10]:
weights = {
      "Dividend Yield(%) Normalised":0.2,
      "Dividend Rate Normalised":0.2,
      "Payout Ratio(%) Normalised":0.2,
      "5 Year Avg Dividend Yield(%) Normalised":0.2,
      "Earning Growth(%) Normalised":0.2
  }
dividend_df["Dividend Score"] = dividend_df[[col for col in weights.keys()]].mul(list(weights.values())).sum(axis=1)

In [12]:
dividend_df

,Ticker,Dividend Yield(%),Dividend Rate,Payout Ratio(%),5 Year Avg Dividend Yield(%),Earning Growth(%),Dividend Yield(%) Normalised,Dividend Rate Normalised,Payout Ratio(%) Normalised,5 Year Avg Dividend Yield(%) Normalised,Earning Growth(%) Normalised,Dividend Score
0,RELIANCE.NS,46.0,6.00,9.210000,44.0,-0.126,0.049238,0.034483,0.903083,0.047146,0.068275,0.220445
1,TCS.NS,564.0,124.00,46.320000,149.0,0.122,0.656506,0.774295,0.512575,0.177419,0.091280,0.442415
2,HDFCBANK.NS,174.0,13.00,21.760000,102.0,0.075,0.199297,0.078370,0.771020,0.119107,0.086920,0.250943
3,INFY.NS,418.0,50.00,64.650000,240.0,0.118,0.485346,0.310345,0.319689,0.290323,0.090909,0.299322
4,ICICIBANK.NS,87.0,11.00,14.710000,99.0,0.084,0.097304,0.065831,0.845207,0.115385,0.087755,0.242296
5,HINDUNILVR.NS,207.0,44.00,95.030000,158.0,0.214,0.237984,0.272727,0.000000,0.188586,0.099814,0.159822
6,SBIN.NS,177.0,17.35,17.440000,150.0,-0.031,0.202814,0.105643,0.816479,0.178660,0.077087,0.276136
7,BAJFINANCE.NS,62.0,5.40,14.420000,24.0,0.214,0.067995,0.030721,0.848258,0.022333,0.099814,0.213824
8,BHARTIARTL.NS,89.0,16.00,36.060002,80.0,-0.340,0.099648,0.097179,0.620541,0.091811,0.048423,0.191520
9,ITC.NS,570.0,16.00,86.920000,372.0,-0.727,0.663540,0.097179,0.085341,0.454094,0.012523,0.262536


In [13]:
dividend_df= dividend_df.sort_values(by="Dividend Score",ascending= False)
dividend_df.head(10)

,Ticker,Dividend Yield(%),Dividend Rate,Payout Ratio(%),5 Year Avg Dividend Yield(%),Earning Growth(%),Dividend Yield(%) Normalised,Dividend Rate Normalised,Payout Ratio(%) Normalised,5 Year Avg Dividend Yield(%) Normalised,Earning Growth(%) Normalised,Dividend Score
45,IOC.NS,720.0,10.0,32.709998,669.0,0.781,0.839390,0.059561,0.655793,0.822581,0.152412,0.505947
31,BPCL.NS,678.0,20.0,37.200000,540.0,0.280,0.790152,0.122257,0.608545,0.662531,0.105937,0.457884
20,ONGC.NS,699.0,18.5,41.000000,511.0,0.478,0.814771,0.112853,0.568557,0.626551,0.124304,0.449407
1,TCS.NS,564.0,124.0,46.320000,149.0,0.122,0.656506,0.774295,0.512575,0.177419,0.091280,0.442415
30,COALINDIA.NS,457.0,22.0,52.320000,812.0,0.129,0.531067,0.134796,0.449437,1.000000,0.091929,0.441446
32,JSWSTEEL.NS,55.0,7.1,3.070000,98.0,9.918,0.059789,0.041379,0.967694,0.114144,1.000000,0.436601
13,HCLTECH.NS,831.0,96.0,88.010000,357.0,-0.002,0.969519,0.598746,0.073871,0.435484,0.079777,0.431480
42,HEROMOTOCO.NS,307.0,150.0,61.040000,313.0,0.257,0.355217,0.937304,0.357677,0.380893,0.103803,0.426979
39,SHREECEM.NS,66.0,160.0,28.970000,33.0,0.034,0.072685,1.000000,0.695149,0.033499,0.083117,0.376890
29,MARUTI.NS,107.0,140.0,28.910000,80.0,-0.064,0.120750,0.874608,0.695780,0.091811,0.074026,0.371395
